In [195]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [196]:
import torch
import torch.nn as nn
import torch.functional as F
import math

from tokenizer import CharacterTokenizer

### Tokenization and Embeddings

Tokenization will be done based on a character-level tokenizer.

In [197]:
text = "Hello! I am learning how to build a transformer from scratch using PyTorch."

tokenizer = CharacterTokenizer(text)

tokens = tokenizer.encode(text)

print(tokenizer.decode(tokens) == text)


True


The dimensions of the embeddings will be
$$
E_{sum}=E_{token}+E_{position}
$$

The shape of these tensors are as follows
- $E_{sum}$ : $x_1 + x_2 \in \mathbb{R^{T \times {d_{embed}}}}$
- $E_{token}$ and $E_{position}$ : $x \in \mathbb{R^{T \times {d_{embed}}}}$

where $T$ is the size of the sequence of tokens


In [198]:
tokens = torch.tensor(tokenizer.encode(text), dtype=torch.long)
d_embed = 64

embedding_matrix = torch.randn(tokenizer.vocab_size, d_embed) / math.sqrt(d_embed)
pos_embedding_matrix = torch.randn(128, d_embed) / math.sqrt(d_embed)
positions = torch.arange(len(tokens))

embeddings = embedding_matrix[tokens]
embedded_tokens = pos_embedding_matrix[positions] + embeddings

print(pos_embedding_matrix.shape)
print(embedded_tokens.shape)
embedded_tokens

torch.Size([128, 64])
torch.Size([75, 64])


tensor([[-0.1100, -0.0238, -0.0433,  ..., -0.2545, -0.0629,  0.1767],
        [-0.1999, -0.1468,  0.0893,  ..., -0.0235, -0.0952, -0.0240],
        [ 0.1836, -0.2943,  0.1075,  ..., -0.0355, -0.2081, -0.3750],
        ...,
        [-0.2638,  0.0256, -0.1175,  ..., -0.0303, -0.0435, -0.0588],
        [ 0.0546,  0.0087,  0.1986,  ..., -0.0022,  0.0679,  0.2344],
        [-0.1186, -0.3935,  0.0448,  ..., -0.1903,  0.1181,  0.1273]])

### Self-Attention Mechanism

We compute the self-attention via computing 3 representations: $W_q, W_k$ and $W_v$



Then, we calculate the scaled dot product and apply a causal mask. The formula is:
$$
S=\frac{QK^T}{\sqrt{d_k}}
$$

After obtaining this result, we apply the softmax multiply by the value vector
$$
Attention(Q, K, V)=Softmax(\frac{QK^T}{\sqrt{d_k}})V
$$

In [199]:
w_q, w_k, w_v = (torch.randn(d_embed, d_embed) / math.sqrt(d_embed), 
                 torch.randn(d_embed, d_embed) / math.sqrt(d_embed), 
                 torch.randn(d_embed, d_embed) / math.sqrt(d_embed))

query_vector = torch.matmul(embedded_tokens, w_q)
key_vector = torch.matmul(embedded_tokens, w_k)
value_vector = torch.matmul(embedded_tokens, w_v)

scaled_dot_product = torch.matmul(query_vector, key_vector.T) / math.sqrt(d_embed)

triu_rows, triu_cols = torch.triu_indices(row=scaled_dot_product.size(0), 
                                  col=scaled_dot_product.size(1),
                                  offset=1)

scaled_dot_product[triu_rows, triu_cols] = float("-inf")

scaled_dot_product.shape
soft_max = nn.Softmax(dim=1)

out = soft_max(scaled_dot_product)
print(out.sum(dim=1) == torch.ones(out.shape[0], out.shape[1]))

A = torch.matmul(out, value_vector)
print(f"Output Shape: {A.shape} \n{A}")



tensor([[ True,  True,  True,  ...,  True, False,  True],
        [ True,  True,  True,  ...,  True, False,  True],
        [ True,  True,  True,  ...,  True, False,  True],
        ...,
        [ True,  True,  True,  ...,  True, False,  True],
        [ True,  True,  True,  ...,  True, False,  True],
        [ True,  True,  True,  ...,  True, False,  True]])
Output Shape: torch.Size([75, 64]) 
tensor([[-0.1891, -0.1314,  0.0526,  ...,  0.1547,  0.0402,  0.1630],
        [-0.1762, -0.0402, -0.1499,  ...,  0.0826, -0.0163,  0.0701],
        [-0.0072, -0.0490, -0.0309,  ...,  0.1330, -0.1230,  0.0855],
        ...,
        [ 0.0159,  0.0099,  0.0125,  ...,  0.0132,  0.0066, -0.0677],
        [ 0.0151,  0.0140,  0.0170,  ...,  0.0154,  0.0064, -0.0655],
        [ 0.0209,  0.0164,  0.0166,  ...,  0.0187,  0.0078, -0.0620]])


### Multi-Head Attention

In [200]:
class MultiHeadAttention(nn.Module):
    def __init__(self, count_heads, d_embed):
        super().__init__()
        self.heads = count_heads
        self.d_embed = d_embed
        self.d_head = d_embed // count_heads

        self.w_q = nn.ParameterList(
            [nn.Parameter(torch.randn(d_embed, self.d_head) / math.sqrt(d_embed)) for _ in range(self.heads)])

        self.w_k = nn.ParameterList(
                [nn.Parameter(torch.randn(d_embed, self.d_head) / math.sqrt(d_embed)) for _ in range(self.heads)])

        self.w_v = nn.ParameterList(
                [nn.Parameter(torch.randn(d_embed, self.d_head) / math.sqrt(d_embed)) for _ in range(self.heads)])
        
        self.wO = nn.Parameter(torch.randn(d_embed, d_embed) / math.sqrt(d_embed))


    def forward(self, X):
        query_vector: list = [torch.matmul(X, w_q) for w_q in self.w_q]
        key_vector: list = [torch.matmul(X, w_k) for w_k in self.w_k]
        value_vector: list = [torch.matmul(X, w_v) for w_v in self.w_v]
        T: int = X.size(0)

        head_outputs = [self.__attention(query_vector[i], key_vector[i], value_vector[i], T) for i in range(self.heads)]
        attention_concat = torch.cat(head_outputs, dim=1)

        return torch.matmul(attention_concat, self.wO)


    def __attention(self, q_i, k_i, v_i, T):
        scaled_dot_product = torch.matmul(q_i, k_i.T) / math.sqrt(self.d_head)

        mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool),
            diagonal=1
        )
        masked_scores = scaled_dot_product.masked_fill(mask, float("-inf"))

        soft_max = nn.Softmax(dim=1)
        out = soft_max(masked_scores)

        attention_matrix = torch.matmul(out, v_i)

        return attention_matrix



In [201]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MultiHeadAttention(4, 64).to(device)
print(model)

out = model(embedded_tokens)
print(f"Output Shape: {out.shape}\n{out}")

MultiHeadAttention(
  (w_q): ParameterList(
      (0): Parameter containing: [torch.float32 of size 64x16]
      (1): Parameter containing: [torch.float32 of size 64x16]
      (2): Parameter containing: [torch.float32 of size 64x16]
      (3): Parameter containing: [torch.float32 of size 64x16]
  )
  (w_k): ParameterList(
      (0): Parameter containing: [torch.float32 of size 64x16]
      (1): Parameter containing: [torch.float32 of size 64x16]
      (2): Parameter containing: [torch.float32 of size 64x16]
      (3): Parameter containing: [torch.float32 of size 64x16]
  )
  (w_v): ParameterList(
      (0): Parameter containing: [torch.float32 of size 64x16]
      (1): Parameter containing: [torch.float32 of size 64x16]
      (2): Parameter containing: [torch.float32 of size 64x16]
      (3): Parameter containing: [torch.float32 of size 64x16]
  )
)
Output Shape: torch.Size([75, 64])
tensor([[ 0.2476, -0.1877, -0.1622,  ...,  0.0382, -0.2765,  0.1234],
        [ 0.0790, -0.1595, -0.137

### Feed-Forward and Layer Normalization

In [202]:
class FeedForward(nn.Module):
    def __init__(self, d_embed):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_embed, 256),
            nn.ReLU(),
            nn.Linear(256, d_embed)
        )
        for layer in self.fc:
            if isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, mean=0.0, std=1 / math.sqrt(layer.in_features))

    def forward(self, X):
        out = self.fc(X)
        return out


class LayerNorm(nn.Module):
    def __init__(self, d_embed, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_embed))
        self.beta = nn.Parameter(torch.zeros(d_embed))
        self.eps = eps

    def forward(self,x):
        mean = torch.mean(x, dim=1, keepdim=True)
        variance = torch.var(x, dim=1, keepdim=True, correction=0)

        return ((x - mean) / torch.sqrt(variance + self.eps)) * self.gamma + self.beta



### Transformer 

In [203]:
class TransformerBlock(nn.Module):
    def __init__(self, num_heads, d_embed):
        super().__init__()
        self.attention = MultiHeadAttention(num_heads, d_embed)
        self.ffn = FeedForward(d_embed)
        
        self.norm1 = LayerNorm(d_embed)
        self.norm2 = LayerNorm(d_embed)

    def forward(self, X):
        x1 = X + self.attention(self.norm1(X))
        x2 = x1 + self.ffn(self.norm2(x1))

        return x2


In [204]:
transform = TransformerBlock(4, 64)
output = transform(embedded_tokens)

print(f"Embedded Tokens Shape: {embedded_tokens.shape}\nTransform Output Shape: {output.shape}")

Embedded Tokens Shape: torch.Size([75, 64])
Transform Output Shape: torch.Size([75, 64])


In [205]:
class AutoRegressiveTransformer(nn.Module):
    def __init__(self, vocab_size, d_embed, num_heads, max_context_length, num_blocks=3):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_heads = num_heads
        self.d_embed = d_embed
        self.d_head = d_embed // num_heads

        self.w_vocab = nn.Parameter(
            torch.randn(d_embed, vocab_size) / math.sqrt(d_embed))

        self.bias = nn.Parameter(
                torch.zeros(vocab_size))

        self.token_embeddings = nn.Parameter(
            torch.randn(vocab_size, d_embed) / math.sqrt(d_embed))
        
        self.pos_embeddings = nn.Parameter(
            torch.randn(max_context_length, d_embed) / math.sqrt(d_embed))

        self.layers = nn.ModuleList(
            [TransformerBlock(self.num_heads, self.d_embed) for _ in range(num_blocks)])

        self.transformer_sequence = nn.Sequential(*self.layers)
        self.layer_norm_final = LayerNorm(d_embed)


    def forward(self, tokens):
        embedded_tokens = self.token_embeddings[tokens] + self.pos_embeddings[:tokens.size(-1)]

        output = self.transformer_sequence(embedded_tokens)
        output = self.layer_norm_final(output)

        logits = torch.matmul(output, self.w_vocab) + self.bias

        return logits;


In [206]:
def create_sequences(tokens, seq_length, start_index=0):
    return (
        tokens[start_index:start_index + seq_length],
        tokens[start_index + 1:start_index + seq_length + 1]
    )

### Example Training

In [207]:
text = "the cat sat on the mat. the cat ate a fish."

tokens = torch.tensor(tokenizer.encode(text))
X, Y = create_sequences(tokens, seq_length = len(tokens) - 1)

vocab_size = tokenizer.vocab_size
d_embed = 64
num_heads = 4
max_context_length = 64
num_blocks = 3
batch_size = 1 # number of sequences

model = AutoRegressiveTransformer(
    vocab_size, d_embed, num_heads, max_context_length, num_blocks)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

In [208]:
def run_training_epoch(X, Y):
    model.train()
    logits = model(X)

    loss = criterion(logits, Y)
    optimizer.zero_grad()

    loss.backward()
    optimizer.step()

    return loss.item()

In [209]:
print(X.shape)
print(Y.shape)
print(model(X).shape)

torch.Size([42])
torch.Size([42])
torch.Size([42, 26])


In [210]:
losses = []

for step in range(200):
    loss = run_training_epoch(X, Y)
    losses.append(loss)

    if step % 20 == 0:
        print(f"Step {step}: Loss = {loss:.4f}")

Step 0: Loss = 3.6182
Step 20: Loss = 0.1072
Step 40: Loss = 0.0218
Step 60: Loss = 0.0121
Step 80: Loss = 0.0090
Step 100: Loss = 0.0073
Step 120: Loss = 0.0061
Step 140: Loss = 0.0053
Step 160: Loss = 0.0046
Step 180: Loss = 0.0040


In [211]:
model.eval()

with torch.no_grad():
    logits = model(X)
    predictions = torch.argmax(logits, dim=-1)

    print("Predicted:", tokenizer.decode(predictions.tolist()))
    print("Expected: ", tokenizer.decode(Y.tolist()))



Predicted: he cat sat on the mat. the cat ate a fish.
Expected:  he cat sat on the mat. the cat ate a fish.


In [212]:

import time

def generate_text(prompt: str, max_new_tokens=50):
    tokens = torch.tensor(tokenizer.encode(prompt), dtype=torch.long)

    model.eval()
    with torch.no_grad():
        print(f"Prompt: {prompt}")
        print(f"Prediction: {prompt}", end="", flush=True)

        time.sleep(1)
        for _ in range(max_new_tokens):

            context = tokens[-max_context_length:]
            logits = model(context)
      
            next_token = torch.argmax(logits[-1])

            # unsqueeze is used to rehsape () to (1, )
            tokens = torch.cat((tokens, next_token.unsqueeze(0)))
            
            print(tokenizer.decode([int(next_token.item())]), end="", flush=True)
            time.sleep(0.05)
    

In [213]:

generate_text("the cat", max_new_tokens=80)

Prompt: the cat
Prediction: the cat sat on the mat. the cat ate a fish. at at fishe ata attt.t.t.the thet.t.t.t.he 